# Market Intelligence Engine — Prototype 3
**Pipeline:** Rule-Based + ML Terarah | **Split:** 60% Train / 40% Test  
**P0 Modules:** Peer Analysis · What Changed · Anomaly Detection · Fundamental Divergence · Opportunity Signal · Risk Signal · Evidence Generation

## 1. Import Library

In [1]:
import os, json, warnings
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

RANDOM_STATE = 42
TRAIN_RATIO  = 0.60          # 60% train / 40% test

# Resolve y_finance_data path relative to notebook location
NB_DIR   = globals().get('__vsc_ipynb_file__', os.getcwd())
if os.path.isfile(NB_DIR):
    NB_DIR = os.path.dirname(NB_DIR)
DATA_DIR = os.path.join(NB_DIR, '..', 'y_finance_data')
DATA_DIR = os.path.abspath(DATA_DIR)
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join(os.getcwd(), 'y_finance_data')

print('Data directory :', DATA_DIR)
print(f'Train / Test   : {TRAIN_RATIO:.0%} / {1-TRAIN_RATIO:.0%}')


Data directory : d:\aduhg\hack\y_finance_data
Train / Test   : 60% / 40%


## 2. Data Loading

In [2]:
EXCLUDE = {'all_stocks_combined.csv', 'scrap.py'}

frames = []
for fname in sorted(os.listdir(DATA_DIR)):
    if fname in EXCLUDE or not fname.endswith('.csv'):
        continue
    tmp = pd.read_csv(os.path.join(DATA_DIR, fname))
    frames.append(tmp)
    print(f'  Loaded {fname}: {len(tmp):,} rows')

df_raw = pd.concat(frames, ignore_index=True)
print(f'\nTotal combined rows: {len(df_raw):,}')
df_raw.head(3)


  Loaded AMRT.csv: 4,338 rows
  Loaded ASII.csv: 6,437 rows
  Loaded BBCA.csv: 5,487 rows
  Loaded GOTO.csv: 1,056 rows
  Loaded TLKM.csv: 5,407 rows

Total combined rows: 22,725


,date,Ticker,open,high,low,close,adj_close,volume,dividends,stock_splits
0,2009-01-15,AMRT,39.5000,42.0000,39.5000,39.5000,31.8562,313145000,0.0000,0.0000
1,2009-01-16,AMRT,40.0000,40.0000,39.5000,40.0000,32.2594,47215000,0.0000,0.0000
2,2009-01-19,AMRT,40.0000,40.0000,39.5000,40.0000,32.2594,11965000,0.0000,0.0000


## 3. Data Cleaning & Feature Engineering

In [3]:
# 3a. Type conversion
df = df_raw.copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce')

NUM_COLS = ['open','high','low','close','adj_close','volume','dividends','stock_splits']
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# 3b. Deduplicate
before = len(df)
df.drop_duplicates(subset=['date','Ticker'], keep='last', inplace=True)
print(f'Duplicates removed: {before - len(df)}')

# 3c. Drop bad dates
df.dropna(subset=['date'], inplace=True)

# 3d. Last 5 years only
cutoff = pd.Timestamp.today() - pd.DateOffset(years=5)
df = df[df['date'] >= cutoff].copy()

# 3e. Remove inf
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 3f. Forward/backward fill per ticker
df.sort_values(['Ticker','date'], inplace=True)
df[NUM_COLS] = df.groupby('Ticker')[NUM_COLS].transform(
    lambda x: x.fillna(method='ffill').fillna(method='bfill')
)
df.dropna(subset=['close'], inplace=True)

print(f'Rows after cleaning (5-year window): {len(df):,}')
print('Tickers:', sorted(df['Ticker'].unique().tolist()))


Duplicates removed: 0
Rows after cleaning (5-year window): 5,858
Tickers: ['AMRT', 'ASII', 'BBCA', 'GOTO', 'TLKM']


In [4]:
# 3g. Feature Engineering

# Returns
df['return_1d']  = df.groupby('Ticker')['close'].pct_change(1)
df['return_5d']  = df.groupby('Ticker')['close'].pct_change(5)
df['return_20d'] = df.groupby('Ticker')['close'].pct_change(20)
df['return_60d'] = df.groupby('Ticker')['close'].pct_change(60)

# Volatility
df['vol_20d'] = df.groupby('Ticker')['return_1d'].transform(lambda x: x.rolling(20).std())
df['vol_60d'] = df.groupby('Ticker')['return_1d'].transform(lambda x: x.rolling(60).std())

# Volume momentum
df['vol_avg_20d'] = df.groupby('Ticker')['volume'].transform(lambda x: x.rolling(20).mean())
df['vol_ratio']   = df['volume'] / (df['vol_avg_20d'] + 1e-9)

# Intraday range
df['intraday_range'] = (df['high'] - df['low']) / (df['close'] + 1e-9)

# Moving averages & price position
df['ma_20']  = df.groupby('Ticker')['close'].transform(lambda x: x.rolling(20).mean())
df['ma_60']  = df.groupby('Ticker')['close'].transform(lambda x: x.rolling(60).mean())
df['ma_200'] = df.groupby('Ticker')['close'].transform(lambda x: x.rolling(200).mean())
df['price_vs_ma20']  = (df['close'] - df['ma_20'])  / (df['ma_20']  + 1e-9)
df['price_vs_ma60']  = (df['close'] - df['ma_60'])  / (df['ma_60']  + 1e-9)
df['price_vs_ma200'] = (df['close'] - df['ma_200']) / (df['ma_200'] + 1e-9)

# RSI-14
def compute_rsi(s, period=14):
    delta = s.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    return 100 - (100 / (1 + gain / (loss + 1e-9)))

df['rsi_14'] = df.groupby('Ticker')['close'].transform(compute_rsi)

# Proxy features
df['div_yield']          = df['dividends'] / (df['close'] + 1e-9)
df['valuation_proxy']    = -df['return_60d']            # contrarian: low momentum = cheaper
df['growth_proxy']       = df['return_20d']
df['institutional_flow'] = np.where(df['vol_ratio'] > 2.0, df['vol_ratio'], 0.0)
df['forecast_proxy']     = df['return_5d'] - df['return_20d']

FEATURE_COLS = [
    'return_1d','return_5d','return_20d','return_60d',
    'vol_20d','vol_60d','vol_ratio','intraday_range',
    'price_vs_ma20','price_vs_ma60','price_vs_ma200',
    'rsi_14','div_yield','valuation_proxy','growth_proxy',
    'institutional_flow','forecast_proxy'
]

df.dropna(subset=FEATURE_COLS, inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Rows after feature engineering: {len(df):,}')
df[FEATURE_COLS].describe()


Rows after feature engineering: 4,863


,return_1d,return_5d,return_20d,return_60d,vol_20d,vol_60d,vol_ratio,intraday_range,price_vs_ma20,price_vs_ma60,price_vs_ma200,rsi_14,div_yield,valuation_proxy,growth_proxy,institutional_flow,forecast_proxy
count,4863.0000,4863.0000,4863.0000,4863.0000,4863.0000,4863.0000,4863.0000,4863.0000,4863.0000,4863.0000,4863.0000,4863.0000,4863.0000,4863.0000,4863.0000,4863.0000,4863.0000
mean,-0.0001,-0.0009,-0.0044,-0.0166,0.0212,0.0221,1.0106,0.0286,-0.0034,-0.0108,-0.0351,48.2090,0.0002,0.0166,-0.0044,0.1814,0.0035
std,0.0239,0.0472,0.0860,0.1399,0.0117,0.0102,0.6925,0.0207,0.0486,0.0790,0.1547,16.5476,0.0031,0.1399,0.0860,0.7685,0.0773
min,-0.2037,-0.2510,-0.3412,-0.5093,0.0000,0.0000,0.0000,0.0000,-0.2268,-0.3394,-0.5524,0.0000,0.0000,-0.5294,-0.3412,0.0000,-0.6290
25%,-0.0116,-0.0246,-0.0512,-0.1076,0.0138,0.0148,0.6193,0.0156,-0.0275,-0.0565,-0.1372,38.0952,0.0000,-0.0709,-0.0512,0.0000,-0.0346
50%,0.0000,0.0000,-0.0045,-0.0203,0.0183,0.0194,0.8610,0.0237,-0.0025,-0.0086,-0.0357,48.7805,0.0000,0.0203,-0.0045,0.0000,0.0027
75%,0.0101,0.0220,0.0393,0.0709,0.0257,0.0273,1.2098,0.0352,0.0223,0.0380,0.0455,58.6957,0.0000,0.1076,0.0393,0.0000,0.0441
max,0.3486,0.3571,0.6964,0.5294,0.0937,0.0607,10.5769,0.3256,0.2980,0.3631,0.5973,100.0000,0.0919,0.5093,0.6964,10.5769,0.4415


## 4. Modeling & Execution (Rule-Based + ML Terarah)

### 4.0  Train / Test Split — Temporal (60% / 40%)

In [5]:
# Label: next-day direction  (1=up, 0=down/flat)
df.sort_values(['Ticker','date'], inplace=True)
df['label'] = (df.groupby('Ticker')['close'].shift(-1) > df['close']).astype(int)
df.dropna(subset=['label'], inplace=True)

# Temporal split — respect time order
unique_dates = sorted(df['date'].unique())
split_idx    = int(len(unique_dates) * TRAIN_RATIO)
split_date   = unique_dates[split_idx]

train_df = df[df['date'] <  split_date].copy()
test_df  = df[df['date'] >= split_date].copy()

X_train, y_train = train_df[FEATURE_COLS], train_df['label']
X_test,  y_test  = test_df[FEATURE_COLS],  test_df['label']

print(f"Train period : {train_df['date'].min().date()} -> {train_df['date'].max().date()}  ({len(train_df):,} rows)")
print(f"Test  period : {test_df['date'].min().date()}  -> {test_df['date'].max().date()}   ({len(test_df):,} rows)")
print(f"Train label dist:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"Test  label dist:\n{y_test.value_counts(normalize=True).round(3)}")


Train period : 2022-07-11 -> 2025-01-02  (2,861 rows)
Test  period : 2025-01-03  -> 2026-09-09   (2,002 rows)
Train label dist:
label
0   0.5760
1   0.4240
Name: proportion, dtype: float64
Test  label dist:
label
0   0.6130
1   0.3870
Name: proportion, dtype: float64


### 4.1  Feature Scaling (StandardScaler — fit on train only)

In [6]:
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print('Scaling complete. Mean~0 / Std~1 verified on training set.')


Scaling complete. Mean~0 / Std~1 verified on training set.


### 4.2  Model Training & Accuracy Evaluation (60/40 Split)

In [7]:
MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest'      : RandomForestClassifier(n_estimators=200, max_depth=8,
                                                   min_samples_leaf=10, random_state=RANDOM_STATE),
}

results = {}
for name, model in MODELS.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    acc    = accuracy_score(y_test, y_pred)
    results[name] = {'model': model, 'acc': acc, 'y_pred': y_pred}
    print(f"\n{'='*55}")
    print(f"Model         : {name}")
    print(f"Test Accuracy (40% hold-out): {acc:.4f}  ({acc*100:.2f}%)")
    print(classification_report(y_test, y_pred, target_names=['Down/Flat', 'Up']))
    print('Confusion Matrix:')
    print(confusion_matrix(y_test, y_pred))

best_name  = max(results, key=lambda k: results[k]['acc'])
best_model = results[best_name]['model']
print(f"\n✅ Best Model Selected : {best_name}  (Acc = {results[best_name]['acc']:.4f})")



Model         : Logistic Regression
Test Accuracy (40% hold-out): 0.5999  (59.99%)
              precision    recall  f1-score   support

   Down/Flat       0.61      0.96      0.75      1228
          Up       0.31      0.03      0.05       774

    accuracy                           0.60      2002
   macro avg       0.46      0.49      0.40      2002
weighted avg       0.49      0.60      0.48      2002

Confusion Matrix:
[[1179   49]
 [ 752   22]]

Model         : Random Forest
Test Accuracy (40% hold-out): 0.6144  (61.44%)
              precision    recall  f1-score   support

   Down/Flat       0.62      0.93      0.75      1228
          Up       0.51      0.11      0.18       774

    accuracy                           0.61      2002
   macro avg       0.57      0.52      0.46      2002
weighted avg       0.58      0.61      0.53      2002

Confusion Matrix:
[[1146   82]
 [ 690   84]]

✅ Best Model Selected : Random Forest  (Acc = 0.6144)


### 4.3  Peer Analysis

In [8]:
# Latest snapshot per ticker
latest = df.sort_values('date').groupby('Ticker').last().reset_index()

PEER_METRICS = ['growth_proxy','valuation_proxy','institutional_flow','rsi_14','vol_ratio']
peer_medians = latest[PEER_METRICS].median()

peer_df = latest[['Ticker'] + PEER_METRICS].copy()
for m in PEER_METRICS:
    peer_df[f'{m}_vs_peer'] = (peer_df[m] - peer_medians[m]).round(6)
    peer_df[f'{m}_rank']    = peer_df[m].rank(ascending=False)

print('Group Medians:')
print(peer_medians.to_string())
print('\nPeer Analysis (latest snapshot):')
peer_df


Group Medians:
growth_proxy          0.0000
valuation_proxy       0.0082
institutional_flow    0.0000
rsi_14               54.1667
vol_ratio             1.1846

Peer Analysis (latest snapshot):


,Ticker,growth_proxy,valuation_proxy,institutional_flow,rsi_14,vol_ratio,growth_proxy_vs_peer,growth_proxy_rank,valuation_proxy_vs_peer,valuation_proxy_rank,institutional_flow_vs_peer,institutional_flow_rank,rsi_14_vs_peer,rsi_14_rank,vol_ratio_vs_peer,vol_ratio_rank
0,AMRT,-0.0542,0.0473,0.0000,41.8182,1.1846,-0.0542,5.0000,0.0391,2.0000,0.0000,3.0000,-12.3485,4.0000,0.0000,3.0000
1,ASII,-0.0041,0.0082,0.0000,54.1667,0.7399,-0.0041,4.0000,0.0000,3.0000,0.0000,3.0000,0.0000,3.0000,-0.4447,5.0000
2,BBCA,0.0235,-0.1013,0.0000,60.9756,1.8431,0.0235,1.0000,-0.1094,5.0000,0.0000,3.0000,6.8089,2.0000,0.6585,1.0000
3,GOTO,0.0000,-0.0000,0.0000,0.0000,1.0929,0.0000,3.0000,-0.0082,4.0000,0.0000,3.0000,-54.1667,5.0000,-0.0917,4.0000
4,TLKM,0.0192,0.0922,0.0000,61.9048,1.3416,0.0192,2.0000,0.0840,1.0000,0.0000,3.0000,7.7381,1.0000,0.1571,2.0000


### 4.4  What Changed? — Period-over-Period Divergence

In [9]:
CHANGE_COLS = ['growth_proxy','valuation_proxy','institutional_flow','forecast_proxy','rsi_14']
wc_records  = []
for ticker, grp in df.groupby('Ticker'):
    grp = grp.sort_values('date')
    if len(grp) < 21:
        continue
    curr  = grp.iloc[-1]
    prior = grp.iloc[-21]   # ~1 calendar month
    rec   = {'Ticker': ticker}
    for col in CHANGE_COLS:
        d = curr[col] - prior[col]
        rec[f'{col}_delta'] = round(d, 6)
        rec[f'{col}_dir']   = '\u2191' if d > 0 else ('\u2193' if d < 0 else '\u2192')
    wc_records.append(rec)

wc_df = pd.DataFrame(wc_records)
print('What Changed (last 20 trading days):')
wc_df


What Changed (last 20 trading days):


,Ticker,growth_proxy_delta,growth_proxy_dir,valuation_proxy_delta,valuation_proxy_dir,institutional_flow_delta,institutional_flow_dir,forecast_proxy_delta,forecast_proxy_dir,rsi_14_delta,rsi_14_dir
0,AMRT,-0.0434,↓,0.0261,↑,0.0000,→,0.0020,↑,-7.5794,↓
1,ASII,-0.0021,↓,-0.1449,↓,0.0000,→,0.0539,↑,18.1667,↑
2,BBCA,-0.0006,↓,-0.0562,↓,0.0000,→,-0.0338,↓,17.7938,↑
3,GOTO,0.0000,→,0.0000,→,0.0000,→,0.0000,→,0.0000,→
4,TLKM,-0.0004,↓,-0.0261,↓,0.0000,→,0.0919,↑,17.9048,↑


### 4.5  Anomaly Detection (IsolationForest per-ticker)

In [10]:
ANOMALY_FEATURES = ['return_1d','vol_ratio','intraday_range','vol_20d']
anom_records     = []

for ticker in df['Ticker'].unique():
    tr = train_df[train_df['Ticker'] == ticker][ANOMALY_FEATURES].dropna()
    al = df[df['Ticker'] == ticker].sort_values('date').copy()
    if len(tr) < 30:
        print(f'  {ticker}: not enough train rows ({len(tr)}) — skipped')
        continue
    iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=RANDOM_STATE)
    iso.fit(tr)
    X_al = al[ANOMALY_FEATURES].fillna(0)
    al['anomaly_score'] = iso.decision_function(X_al)
    al['is_anomaly']    = iso.predict(X_al) == -1
    lr = al.iloc[-1]
    anom_records.append({
        'Ticker'       : ticker,
        'date'         : lr['date'],
        'anomaly_score': round(float(lr['anomaly_score']), 4),
        'is_anomaly'   : bool(lr['is_anomaly']),
        'return_1d'    : round(float(lr['return_1d']), 4),
        'vol_ratio'    : round(float(lr['vol_ratio']), 4),
    })

anomaly_df = pd.DataFrame(anom_records)
print('Anomaly Detection Results (latest per ticker):')
anomaly_df


Anomaly Detection Results (latest per ticker):


,Ticker,date,anomaly_score,is_anomaly,return_1d,vol_ratio
0,AMRT,2026-09-09,0.1645,False,0.0000,1.1846
1,ASII,2026-09-09,0.1948,False,-0.0162,0.7399
2,BBCA,2026-09-09,-0.0006,True,-0.0225,1.8431
3,GOTO,2026-09-09,0.0419,False,0.0000,1.0929
4,TLKM,2026-09-09,0.0924,False,0.0038,1.3416


### 4.6  Fundamental Divergence Detection

In [11]:
div_records = []
for ticker, grp in df.groupby('Ticker'):
    grp = grp.sort_values('date')
    if len(grp) < 21:
        continue
    curr  = grp.iloc[-1]
    prior = grp.iloc[-21]
    g_up = curr['growth_proxy']       > prior['growth_proxy']
    f_up = curr['forecast_proxy']     > prior['forecast_proxy']
    i_up = curr['institutional_flow'] > prior['institutional_flow']
    v_dn = curr['valuation_proxy']    < prior['valuation_proxy']

    sup = []
    if g_up: sup.append('Revenue/Growth improved (\u2191)')
    if f_up: sup.append('Future Forecast improved (\u2191)')
    if i_up: sup.append('Institutional Flow increased (\u2191)')
    if v_dn: sup.append('Valuation compressed vs prior (\u2193 = cheaper)')

    detected  = (g_up or f_up or i_up) and v_dn
    up_cnt    = sum([g_up, f_up, i_up, v_dn])
    conf      = 'High' if up_cnt >= 3 else ('Medium' if up_cnt == 2 else 'Low')
    div_records.append({'Ticker':ticker,'detected':detected,'confidence':conf,'supporting_factors':sup})

divergence_df = pd.DataFrame(div_records)
print('Fundamental Divergence:')
divergence_df[['Ticker','detected','confidence']]


Fundamental Divergence:


,Ticker,detected,confidence
0,AMRT,False,Low
1,ASII,True,Medium
2,BBCA,False,Low
3,GOTO,False,Low
4,TLKM,True,Medium


### 4.7  Opportunity & Risk Scoring

In [12]:
# Weights per plan: Growth(25%) Valuation(25%) Institutional(20%) Forecast(20%) Peer(10%)
SCORE_FEATURES = ['growth_proxy','valuation_proxy','institutional_flow','forecast_proxy']
WEIGHTS = {
    'growth_proxy'      : 0.25,
    'valuation_proxy'   : 0.25,
    'institutional_flow': 0.20,
    'forecast_proxy'    : 0.20,
    'peer_position'     : 0.10,
}

score_scaler = StandardScaler()
score_scaler.fit(train_df[SCORE_FEATURES].fillna(0))

ls = latest.copy()
z_mat = score_scaler.transform(ls[SCORE_FEATURES].fillna(0))
z_df  = pd.DataFrame(z_mat, columns=SCORE_FEATURES, index=ls.index)
z_df['peer_position'] = (ls['growth_proxy'].rank() / len(ls) - 0.5) * 2  # normalised -1..+1

ls['opp_score'] = 50.0
for feat, w in WEIGHTS.items():
    ls['opp_score'] += z_df[feat] * w * 25     # each unit Z contributes proportionally

ls['opp_score']  = ls['opp_score'].clip(1, 99).round(1)
ls['direction']  = np.where(ls['opp_score'] > 50, 'Bullish', 'Bearish')
ls['risk']       = pd.cut(ls['vol_20d'],
                          bins=[0, 0.01, 0.02, np.inf],
                          labels=['Low','Medium','High'])
dist = (ls['opp_score'] - 50).abs()
ls['confidence'] = pd.cut(dist, bins=[0, 10, 20, np.inf], labels=['Low','Medium','High'])

latest_scored = ls
print('Opportunity & Risk Scores:')
latest_scored[['Ticker','opp_score','direction','confidence','risk']]


Opportunity & Risk Scores:


,Ticker,opp_score,direction,confidence,risk
0,AMRT,49.3000,Bearish,Low,High
1,ASII,48.8000,Bearish,Low,Medium
2,BBCA,45.5000,Bearish,Low,Medium
3,GOTO,49.4000,Bearish,Low,NaN
4,TLKM,56.4000,Bullish,Low,Low


### 4.8  Evidence Generation

In [13]:
ev_records = []
for _, row in latest_scored.iterrows():
    ticker = row['Ticker']
    pos, neg, ev = [], [], []

    # Growth
    gp_c = round(row['growth_proxy']*100, 2)
    gp_p = round(peer_medians['growth_proxy']*100, 2)
    if row['growth_proxy'] > peer_medians['growth_proxy']:
        pos.append('Growth above peer median')
    else:
        neg.append('Growth below peer median')
    ev.append(f'Growth: {gp_c}% vs peer {gp_p}%')

    # Valuation
    vp_c = round(row['valuation_proxy']*100, 2)
    vp_p = round(peer_medians['valuation_proxy']*100, 2)
    if row['valuation_proxy'] > peer_medians['valuation_proxy']:
        pos.append('Valuation attractive vs peer (\u2193 60d momentum = cheaper)')
    else:
        neg.append('Valuation stretched vs peer')
    ev.append(f'Valuation proxy: {vp_c} vs peer {vp_p}')

    # Institutional flow
    if_c = round(row['institutional_flow'], 2)
    if_p = round(peer_medians['institutional_flow'], 2)
    if row['institutional_flow'] > peer_medians['institutional_flow']:
        pos.append('Institutional flow above peer')
    else:
        neg.append('Institutional flow below peer')
    ev.append(f'Institutional Flow: {if_c}x vs peer {if_p}x')

    # RSI
    rsi = round(row['rsi_14'], 1)
    if rsi > 70:
        neg.append(f'RSI overbought ({rsi})')
        ev.append(f'RSI-14: {rsi} (OB threshold = 70)')
    elif rsi < 30:
        pos.append(f'RSI oversold — potential reversal ({rsi})')
        ev.append(f'RSI-14: {rsi} (OS threshold = 30)')
    else:
        ev.append(f'RSI-14: {rsi} (Neutral zone)')

    # Dividends
    div = round(row['div_yield']*100, 4)
    if row['dividends'] > 0:
        pos.append(f'Active dividend payer (yield = {div}%)')
    else:
        neg.append('No recent dividend payment')

    ev_records.append({'Ticker':ticker,'positive_factors':pos,'negative_factors':neg,'evidence':ev})

evidence_df = pd.DataFrame(ev_records)
print('Evidence Generation complete.')
evidence_df[['Ticker','positive_factors','negative_factors']]


Evidence Generation complete.


,Ticker,positive_factors,negative_factors
0,AMRT,[Valuation attractive vs peer (↓ 60d momentum ...,"[Growth below peer median, Institutional flow ..."
1,ASII,[],"[Growth below peer median, Valuation stretched..."
2,BBCA,[Growth above peer median],"[Valuation stretched vs peer, Institutional fl..."
3,GOTO,[RSI oversold — potential reversal (0.0)],"[Growth below peer median, Valuation stretched..."
4,TLKM,"[Growth above peer median, Valuation attractiv...","[Institutional flow below peer, No recent divi..."


## 5. Structured JSON Output

In [14]:
timestamp = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S')
output    = []

for ticker in sorted(df['Ticker'].unique()):
    sr = latest_scored[latest_scored['Ticker'] == ticker]
    ar = anomaly_df[anomaly_df['Ticker'] == ticker]
    dr = divergence_df[divergence_df['Ticker'] == ticker]
    er = evidence_df[evidence_df['Ticker'] == ticker]
    if sr.empty:
        continue
    s = sr.iloc[0]
    a = ar.iloc[0] if not ar.empty else {}
    d = dr.iloc[0] if not dr.empty else {}
    e = er.iloc[0] if not er.empty else {}

    record = {
        'ticker': ticker,
        'intelligence_output': {
            'anomaly'         : bool(a.get('is_anomaly', False)),
            'direction'       : str(s['direction']),
            'score'           : float(s['opp_score']),
            'confidence'      : str(s['confidence']),
            'risk'            : str(s['risk']),
            'positive_factors': list(e.get('positive_factors', [])),
            'negative_factors': list(e.get('negative_factors', [])),
        },
        'fundamental_divergence': {
            'detected'          : bool(d.get('detected', False)),
            'confidence'        : str(d.get('confidence', 'Low')),
            'supporting_factors': list(d.get('supporting_factors', [])),
        },
        'evidence' : list(e.get('evidence', [])),
        'timestamp': timestamp,
    }
    output.append(record)

print(json.dumps(output, indent=2, ensure_ascii=False))


[
  {
    "ticker": "AMRT",
    "intelligence_output": {
      "anomaly": false,
      "direction": "Bearish",
      "score": 49.3,
      "confidence": "Low",
      "risk": "High",
      "positive_factors": [
        "Valuation attractive vs peer (↓ 60d momentum = cheaper)"
      ],
      "negative_factors": [
        "Growth below peer median",
        "Institutional flow below peer",
        "No recent dividend payment"
      ]
    },
    "fundamental_divergence": {
      "detected": false,
      "confidence": "Low",
      "supporting_factors": [
        "Future Forecast improved (↑)"
      ]
    },
    "evidence": [
      "Growth: -5.42% vs peer 0.0%",
      "Valuation proxy: 4.73 vs peer 0.82",
      "Institutional Flow: 0.0x vs peer 0.0x",
      "RSI-14: 41.8 (Neutral zone)"
    ],
    "timestamp": "2026-09-12T17:17:28"
  },
  {
    "ticker": "ASII",
    "intelligence_output": {
      "anomaly": false,
      "direction": "Bearish",
      "score": 48.8,
      "confidence": "Low",
 

## 6. Model Accuracy Summary (60% Train / 40% Test)

In [15]:
print('=' * 60)
print('MODEL ACCURACY SUMMARY — Temporal 60/40 Split')
print('=' * 60)
for name, res in results.items():
    marker = '<-- BEST' if name == best_name else ''
    print(f"  {name:<30} Acc: {res['acc']:.4f}  ({res['acc']*100:.2f}%)  {marker}")
print()
print(f'  Best Model : {best_name}')
print(f'  Best Acc   : {results[best_name]["acc"]:.4f}')
print('=' * 60)
print('\nFinal Signal Summary per Ticker:')
for rec in output:
    io = rec['intelligence_output']
    fd = rec['fundamental_divergence']
    print(
        f"  [{rec['ticker']}]  Score={io['score']:<5}  Dir={io['direction']:<8} "
        f"Conf={io['confidence']:<7} Risk={io['risk']:<7} "
        f"Anomaly={io['anomaly']}  FundDiv={fd['detected']}"
    )


MODEL ACCURACY SUMMARY — Temporal 60/40 Split
  Logistic Regression            Acc: 0.5999  (59.99%)  
  Random Forest                  Acc: 0.6144  (61.44%)  <-- BEST

  Best Model : Random Forest
  Best Acc   : 0.6144

Final Signal Summary per Ticker:
  [AMRT]  Score=49.3   Dir=Bearish  Conf=Low     Risk=High    Anomaly=False  FundDiv=False
  [ASII]  Score=48.8   Dir=Bearish  Conf=Low     Risk=Medium  Anomaly=False  FundDiv=True
  [BBCA]  Score=45.5   Dir=Bearish  Conf=Low     Risk=Medium  Anomaly=True  FundDiv=False
  [GOTO]  Score=49.4   Dir=Bearish  Conf=Low     Risk=nan     Anomaly=False  FundDiv=False
  [TLKM]  Score=56.4   Dir=Bullish  Conf=Low     Risk=Low     Anomaly=False  FundDiv=True
